In [ ]:
from __future__ import print_function

import sys
import random
import ftplib
from datetime import datetime, timedelta
from io import BytesIO

import pandas as pd
import pytz

sys.path.append("/var/www/python/Prod/nighthawk/")
from nighthawk.util import sql_functions, dataframe_functions

sys.path.append("/var/www/python/Prod/COMMON/monitoring/notification_system/")
from send_msg_through_slack import send_msg_through_slack

sys.path.append("/var/www/python/Prod/SPP/DataDownload/LMP/")
from spp_update_lmp_summary import run_cron_job_summaries

# Config
FTP_HOST = 'pubftp-mte.itespp.org'
FTP_BASE = '/Markets/DA/LMP_By_SETTLEMENT_LOC'
# Per-day file path under FTP_BASE: YYYY/MM/By_Day/DA-LMP-SL-YYYYMMDD0100.csv
TZ = pytz.timezone('America/Chicago')



In [3]:
"""
:Author: Qingcheng Wei
Date: 03/04/2026


Objective:
    Download SPP Mid-Term Wind Forecast (MTWF) and Mid-Term Solar Forecast (MTSF) data.
    Parse ONLY the LATEST downloaded CSV file and update the database tables:

    Intermediate table:
    - spp_physical.MTWF                               (added baa_zone; PK: ForecastTime, IntervalEnd, baa_zone)

    Per-BAA output tables:
    - spp_physical.baa_zonal_latest_wind_forecast     (latest wind forecast per BAA zone; PK: dt, hr, baa_zone)
    - spp_physical.baa_zonal_virtual_wind_forecast    (virtual wind forecast per BAA zone; PK: dt, hr, baa_zone)
    - spp_physical.baa_zonal_latest_solar_forecast    (latest solar forecast per BAA zone; PK: dt, hr, baa_zone)
    - spp_physical.baa_zonal_virtual_solar_forecast   (virtual solar forecast per BAA zone; PK: dt, hr, baa_zone)

    Market-total output tables:
    - spp_physical.spp_latest_wind_forecast           (latest wind forecast summed across all BAA zones; PK: dt, hr)
    - spp_physical.spp_virtual_wind_forecast          (virtual wind forecast summed across all BAA zones; PK: dt, hr)
    - spp_physical.spp_latest_solar_forecast          (latest solar forecast summed across all BAA zones; PK: dt, hr)
    - spp_physical.spp_virtual_solar_forecast         (virtual solar forecast summed across all BAA zones; PK: dt, hr)

Data Source: ftp://pubftp.spp.org/Operational_Data/MTRF/

Calling this program:
    python spp_dataDownload_MTWF.py [START_DATE] [END_DATE]
    If dates are not provided, it defaults to yesterday and today.
"""

import sys
import os
import glob
import pandas as pd
import datetime
import pytz
import subprocess
import shutil

# Add project path for custom modules
sys.path.append("/var/www/python/Prod/nighthawk/")
from nighthawk.util import connections, sql_functions

sys.path.append("/var/www/python/Prod/COMMON/monitoring/notification_system/")
from send_msg_through_slack import send_msg_through_slack

script_name= 'spp_dataDownload_MTWF.py'

# Define the set of columns that are essential for the script to function.
ESSENTIAL_MTWF_COLUMNS = {
    'Interval',
    'GMTIntervalEnd',
    'Wind Forecast MW',
    'Solar Forecast MW'
}

# When BAA column is present, map its value to the baa_zone code stored in the DB.
# Old-format files that lack BAA are treated as SPP (East).
BAA_ZONE_MAP = {
    'SPP':  'E',
    'SWPW': 'W',
}


def log_message(level, type, line, message):
    print(f"{level} - {type} - Line {line}: {message}")

def download_forecast(dt):
    try:
        download_folder = "/var/www/python/Qingcheng/"
        if not os.path.exists(download_folder):
            try:
                os.makedirs(download_folder)
            except OSError as e:
                print(f"Error creating directory {download_folder}: {e}")
                return

        dt_str = dt.strftime("%Y/%m/%d")
        ftp_url = f"ftp://pubftp-mte.itespp.org/Operational_Data/MTRF/{dt_str}"

        # Using wget to download files recursively, no directories, no clobber
        cmd = f"wget -nd -r -nc {ftp_url} -P {download_folder}"
        print(f"Executing: {cmd}")
        subprocess.run(cmd, shell=True)

        # List files in the directory
        files = glob.glob(os.path.join(download_folder, "*"))

        # Filter files based on name pattern
        mtlf_files = [f for f in files if ('MTRF' in os.path.basename(f) or 'DAWF' in os.path.basename(f) or 'MTWF' in os.path.basename(f))]

        if not mtlf_files:
            print('no file been found')
            return

        mtlf_files.sort(key=lambda x: ''.join(filter(str.isdigit, os.path.basename(x))))
        # Now that it's sorted, the LATEST timestamp is the LAST item
        latest_file = mtlf_files[7]
        print(f"\nLatest file identified: {os.path.basename(latest_file)}")

        # Process ONLY the latest file
        process_file(latest_file)

    finally:
        if os.path.exists(download_folder):
            shutil.rmtree(download_folder)
            print(f"Cleanup: Folder {download_folder} has been removed.")


def process_file(file_path):
    global warning_string
    file_name = os.path.basename(file_path)
    print(f"\n--- Processing File: {file_name} ---")

    # Extract time from filename (digits only) + "00"
    time_of_file = ''.join(filter(str.isdigit, file_name))
    time_of_file += "00"
    print(f"time of file is {time_of_file}")

    # Read and clean column names
    df = pd.read_csv(file_path)
    df.columns = [c.strip() for c in df.columns]

    actual_columns = set(df.columns)
    expected_columns=ESSENTIAL_MTWF_COLUMNS
    known_optional_columns = {'BAA'}

    # Check for column mismatches and handle them.
    if not expected_columns.issubset(actual_columns):
        missing_cols = expected_columns - actual_columns
        exception_str = (
            f"\n[ERROR] File '{file_path}' has schema mismatch. "
            f"Missing essential columns: {missing_cols}. "
            f"Skipping file.\n"
        )
        warning_string += exception_str
        print(exception_str.strip())
        return

    truly_new_cols = actual_columns - expected_columns - known_optional_columns
    if truly_new_cols:
        warning_string += f"\n[INFO] File '{file_path}' has new, unexpected columns: {truly_new_cols}.\n"

    
    # 1. Date Conversion (Vectorized)
    file_date = pd.to_datetime(time_of_file[:8], format='%Y%m%d').date()
    next_day  = file_date + datetime.timedelta(days=1)
    df['IntervalEnd'] = pd.to_datetime(df['Interval'], errors='coerce')
    df['GMTIntervalEnd'] = pd.to_datetime(df['GMTIntervalEnd'], errors='coerce')
    df = df[df['IntervalEnd'].dt.date == next_day].copy()
    print(df.head())
    return 
    # Drop rows where date parsing failed
    df.dropna(subset=['IntervalEnd', 'GMTIntervalEnd'], inplace=True)

    # 2. Process and Coalesce Wind Forecast Data (Vectorized)
    df['MTWF'] = pd.to_numeric(df.get('Wind Forecast MW'), errors='coerce').fillna(0)

    # 3. Process Solar Forecast Data (Vectorized)
    df['MTSF'] = pd.to_numeric(df.get('Solar Forecast MW'), errors='coerce').fillna(0)

    # 4. Add ForecastTime
    df['ForecastTime'] = pd.to_datetime(time_of_file, format='%Y%m%d%H%M%S')

    # 5. Derive baa_zone: 'E' for SPP, 'W' for SWPW.
    # Old-format files that lack the BAA column default to 'E' (SPP only).
    if 'BAA' in df.columns:
        df['baa_zone'] = df['BAA'].str.strip().map(BAA_ZONE_MAP)
        unknown_baa = df['baa_zone'].isna() & df['BAA'].notna()
        if unknown_baa.any():
            bad_vals = df.loc[unknown_baa, 'BAA'].unique().tolist()
            warning_string += (
                f"\n[WARNING] File '{file_path}' contains unrecognised BAA values: {bad_vals}. "
                f"Rows will be skipped.\n"
            )
        df.dropna(subset=['baa_zone'], inplace=True)
    else:
        df['baa_zone'] = 'E'

    # 6. Filtering (Vectorized)
    df_filtered = df[df['IntervalEnd'].dt.year > 2000].copy()

    # 7. Final Formatting and Upload
    df_filtered['IntervalEnd'] = df_filtered['IntervalEnd'].dt.strftime('%Y-%m-%d %H:%M:%S')
    df_filtered['GMTIntervalEnd'] = df_filtered['GMTIntervalEnd'].dt.strftime('%Y-%m-%d %H:%M:%S')
    df_filtered['ForecastTime'] = df_filtered['ForecastTime'].dt.strftime('%Y-%m-%d %H:%M:%S')

    df_upload = df_filtered[['ForecastTime', 'IntervalEnd', 'GMTIntervalEnd', 'MTWF', 'MTSF', 'baa_zone']]
    print(f"Rows found: {len(df_upload)}")

    sql_functions.replace_into_sql_table(
            df=df_upload,
            final_table_fullname='spp_temp.MTWF_mte',
            print_details=True
        )


def update_latest_tables(start_dt,lookback_days=20):
    start_time=start_dt-datetime.timedelta(days=lookback_days)
    # Update latest wind forecast
    sql_functions.run_sql_query("""
        CREATE TABLE IF NOT EXISTS spp_physical.baa_zonal_latest_wind_forecast (
         `dt` date NOT NULL,
         `hr` tinyint(3) unsigned NOT NULL,
         `baa_zone` char(10) NOT NULL DEFAULT 'E',
         `Gen_MW` float(8,2) DEFAULT NULL,
         PRIMARY KEY (`dt`,`hr`,`baa_zone`)
        ) ENGINE=MyISAM
    """)

    query_wind_latest = f"""
        REPLACE INTO spp_physical.baa_zonal_latest_wind_forecast
        (dt,hr,baa_zone,Gen_MW)
        SELECT date(w.IntervalEnd), hour(w.IntervalEnd)+1, w.baa_zone, NULLIF(w.MTWF, 0)
        FROM
        spp_physical.MTWF w
        JOIN (SELECT IntervalEnd, baa_zone, max(ForecastTime) ForecastTime
        FROM spp_physical.MTWF WHERE IntervalEnd > Date('{start_time.date()}') GROUP BY IntervalEnd, baa_zone) ft
            ON w.IntervalEnd=ft.IntervalEnd AND w.baa_zone=ft.baa_zone AND w.ForecastTime=ft.ForecastTime
    """
    sql_functions.run_sql_query(query_wind_latest)
    print("\ndata updated in latest wind table")

    sql_functions.run_sql_query("""
        CREATE TABLE IF NOT EXISTS spp_physical.spp_latest_wind_forecast (
         `dt` date NOT NULL,
         `hr` tinyint(3) unsigned NOT NULL,
         `Gen_MW` float(8,2) DEFAULT NULL,
         PRIMARY KEY (`dt`,`hr`)
        ) ENGINE=MyISAM
    """)

    query_wind_latest_total = f"""
        REPLACE INTO spp_physical.spp_latest_wind_forecast
        (dt,hr,Gen_MW)
        SELECT dt, hr, NULLIF(SUM(Gen_MW), 0)
        FROM spp_physical.baa_zonal_latest_wind_forecast
        WHERE dt > Date('{start_time.date()}')
        GROUP BY dt, hr
    """
    sql_functions.run_sql_query(query_wind_latest_total)
    print("\ndata updated in latest wind total table")

    # Update virtual wind forecast
    sql_functions.run_sql_query("""
        CREATE TABLE IF NOT EXISTS spp_physical.baa_zonal_virtual_wind_forecast (
         `dt` date NOT NULL,
         `hr` tinyint(3) unsigned NOT NULL,
         `baa_zone` char(10) NOT NULL DEFAULT 'E',
         `Gen_MW` float(8,2) DEFAULT NULL,
         PRIMARY KEY (`dt`,`hr`,`baa_zone`)
        ) ENGINE=MyISAM
    """)

    query_wind_virtual = f"""
        REPLACE INTO spp_physical.baa_zonal_virtual_wind_forecast
        (dt,hr,baa_zone,Gen_MW)
        SELECT date(w.IntervalEnd), hour(w.IntervalEnd)+1, w.baa_zone, NULLIF(w.MTWF, 0)
        FROM
        spp_physical.MTWF w
        JOIN (SELECT IntervalEnd, baa_zone, max(ForecastTime) ForecastTime
        FROM spp_physical.MTWF WHERE ForecastTime <= date(IntervalEnd) + INTERVAL 8 HOUR - INTERVAL 1 DAY and IntervalEnd >  Date('{start_time.date()}') GROUP BY IntervalEnd, baa_zone) ft
            ON w.IntervalEnd=ft.IntervalEnd AND w.baa_zone=ft.baa_zone AND w.ForecastTime=ft.ForecastTime
    """
    sql_functions.run_sql_query(query_wind_virtual)
    print("\ndata updated in virtual wind table")

    sql_functions.run_sql_query("""
        CREATE TABLE IF NOT EXISTS spp_physical.spp_virtual_wind_forecast (
         `dt` date NOT NULL,
         `hr` tinyint(3) unsigned NOT NULL,
         `Gen_MW` float(8,2) DEFAULT NULL,
         PRIMARY KEY (`dt`,`hr`)
        ) ENGINE=MyISAM
    """)

    query_wind_virtual_total = f"""
        REPLACE INTO spp_physical.spp_virtual_wind_forecast
        (dt,hr,Gen_MW)
        SELECT dt, hr, NULLIF(SUM(Gen_MW), 0)
        FROM spp_physical.baa_zonal_virtual_wind_forecast
        WHERE dt > Date('{start_time.date()}')
        GROUP BY dt, hr
    """
    sql_functions.run_sql_query(query_wind_virtual_total)
    print("\ndata updated in virtual wind total table")

    # Update latest solar forecast
    sql_functions.run_sql_query("""
        CREATE TABLE IF NOT EXISTS spp_physical.baa_zonal_latest_solar_forecast (
         `dt` date NOT NULL,
         `hr` tinyint(3) unsigned NOT NULL,
         `baa_zone` char(10) NOT NULL DEFAULT 'E',
         `Gen_MW` float(8,2) DEFAULT NULL,
         PRIMARY KEY (`dt`,`hr`,`baa_zone`)
        ) ENGINE=MyISAM
    """)

    query_solar_latest = f"""
        REPLACE INTO spp_physical.baa_zonal_latest_solar_forecast
        (dt,hr,baa_zone,Gen_MW)
        SELECT date(w.IntervalEnd), hour(w.IntervalEnd)+1, w.baa_zone, NULLIF(w.MTSF, 0)
        FROM
        spp_physical.MTWF w
        JOIN (SELECT IntervalEnd, baa_zone, max(ForecastTime) ForecastTime
        FROM spp_physical.MTWF WHERE IntervalEnd > Date('{start_time.date()}') GROUP BY IntervalEnd, baa_zone) ft
            ON w.IntervalEnd=ft.IntervalEnd AND w.baa_zone=ft.baa_zone AND w.ForecastTime=ft.ForecastTime
    """
    sql_functions.run_sql_query(query_solar_latest)
    print("\ndata updated in latest solar table")

    sql_functions.run_sql_query("""
        CREATE TABLE IF NOT EXISTS spp_physical.spp_latest_solar_forecast (
         `dt` date NOT NULL,
         `hr` tinyint(3) unsigned NOT NULL,
         `Gen_MW` float(8,2) DEFAULT NULL,
         PRIMARY KEY (`dt`,`hr`)
        ) ENGINE=MyISAM
    """)

    query_solar_latest_total = f"""
        REPLACE INTO spp_physical.spp_latest_solar_forecast
        (dt,hr,Gen_MW)
        SELECT dt, hr, NULLIF(SUM(Gen_MW), 0)
        FROM spp_physical.baa_zonal_latest_solar_forecast
        WHERE dt > Date('{start_time.date()}')
        GROUP BY dt, hr
    """
    sql_functions.run_sql_query(query_solar_latest_total)
    print("\ndata updated in latest solar total table")

    # Update virtual solar forecast
    sql_functions.run_sql_query("""
        CREATE TABLE IF NOT EXISTS spp_physical.baa_zonal_virtual_solar_forecast (
         `dt` date NOT NULL,
         `hr` tinyint(3) unsigned NOT NULL,
         `baa_zone` char(10) NOT NULL DEFAULT 'E',
         `Gen_MW` float(8,2) DEFAULT NULL,
         PRIMARY KEY (`dt`,`hr`,`baa_zone`)
        ) ENGINE=MyISAM
    """)

    query_solar_virtual = f"""
        REPLACE INTO spp_physical.baa_zonal_virtual_solar_forecast
        (dt,hr,baa_zone,Gen_MW)
        SELECT date(w.IntervalEnd), hour(w.IntervalEnd)+1, w.baa_zone, NULLIF(w.MTSF, 0)
        FROM
        spp_physical.MTWF w
        JOIN (SELECT IntervalEnd, baa_zone, max(ForecastTime) ForecastTime
        FROM spp_physical.MTWF WHERE ForecastTime <= date(IntervalEnd) + INTERVAL 8 HOUR - INTERVAL 1 DAY and IntervalEnd > Date('{start_time.date()}') GROUP BY IntervalEnd, baa_zone) ft
            ON w.IntervalEnd=ft.IntervalEnd AND w.baa_zone=ft.baa_zone AND w.ForecastTime=ft.ForecastTime
    """
    sql_functions.run_sql_query(query_solar_virtual)
    print("\ndata updated in virtual solar table")

    sql_functions.run_sql_query("""
        CREATE TABLE IF NOT EXISTS spp_physical.spp_virtual_solar_forecast (
         `dt` date NOT NULL,
         `hr` tinyint(3) unsigned NOT NULL,
         `Gen_MW` float(8,2) DEFAULT NULL,
         PRIMARY KEY (`dt`,`hr`)
        ) ENGINE=MyISAM
    """)

    query_solar_virtual_total = f"""
        REPLACE INTO spp_physical.spp_virtual_solar_forecast
        (dt,hr,Gen_MW)
        SELECT dt, hr, NULLIF(SUM(Gen_MW), 0)
        FROM spp_physical.baa_zonal_virtual_solar_forecast
        WHERE dt > Date('{start_time.date()}')
        GROUP BY dt, hr
    """
    sql_functions.run_sql_query(query_solar_virtual_total)
    print("\ndata updated in virtual solar total table")


def check_missing_latest_wind(start_dt,lookback_days=14):
    """
    Checks all four total forecast tables (wind latest, wind virtual, solar latest,
    solar virtual) for missing dt/hr combinations. Returns a unique list of dates
    where any hour is absent from any of the four tables.
    """
    print(f"\n--- Checking total wind/solar forecast tables for gaps (Last {lookback_days} days) ---")
    start_time = start_dt - datetime.timedelta(days=lookback_days)

    # 1. Generate the reference grid of every (dt, hr) that should exist
    end_date = start_dt - datetime.timedelta(hours=1)
    full_range = pd.date_range(start=start_time, end=end_date, freq='h')
    reference_df = pd.DataFrame({
        'dt': full_range.date,
        'hr': full_range.hour + 1
    })

    # 2. Pull existing (dt, hr) pairs from each total table
    def fetch_table(table_name):
        query = f"""
            SELECT dt, hr
            FROM {table_name}
            WHERE dt >= Date('{start_time.date()}')
        """
        df = sql_functions.download_df_from_sql_db(query)
        df['dt'] = pd.to_datetime(df['dt']).dt.date
        return df

    tables = [
        'spp_physical.spp_latest_wind_forecast',
        'spp_physical.spp_virtual_wind_forecast',
        'spp_physical.spp_latest_solar_forecast',
        'spp_physical.spp_virtual_solar_forecast',
    ]

    # 3. Union all missing (dt, hr) slots across every table
    missing_frames = []
    for table in tables:
        existing = fetch_table(table)
        merged = pd.merge(reference_df, existing, on=['dt', 'hr'], how='left', indicator=True)
        missing_frames.append(merged[merged['_merge'] == 'left_only'][['dt', 'hr']])

    df_missing = pd.concat(missing_frames).drop_duplicates()

    # 4. Reporting
    if df_missing.empty:
        print("All total wind/solar forecast tables are 100% complete.")
        return []

    missing_dates = sorted(df_missing['dt'].unique())
    print(f"Found {len(df_missing)} missing hourly slots across one or more total tables.")
    print(f"Dates requiring backfill: {[d.strftime('%Y-%m-%d') for d in missing_dates]}")
    return missing_dates

if __name__ == "__main__":
    # Set timezone
    timezone = pytz.timezone("America/Chicago")

    log_message("Debug", "Information", sys._getframe().f_lineno, "Starting Script\n")

   
    # Ensure main table exists
    sql_functions.run_sql_query("""
            CREATE TABLE IF NOT EXISTS spp_temp.MTWF_mte (
             `ForecastTime` datetime NOT NULL,
             `IntervalEnd` datetime NOT NULL,
             `GMTIntervalEnd` datetime DEFAULT NULL,
             `MTWF` float(10,2) NOT NULL DEFAULT 0.0,
             `MTSF` float(10,2) NOT NULL DEFAULT 0.0,
             `baa_zone` char(10) NOT NULL DEFAULT 'E',
             PRIMARY KEY (`ForecastTime`,`IntervalEnd`,`baa_zone`)
            ) ENGINE=MyISAM
        """)

        # Loop through dates
    current_dt = pd.to_datetime('2026-03-16')
    download_forecast(current_dt)




    # # Update derived tables
    # update_latest_tables(start_dt)






Debug - Information - Line 436: Starting Script

Executing: wget -nd -r -nc ftp://pubftp-mte.itespp.org/Operational_Data/MTRF/2026/03/16 -P /var/www/python/Qingcheng/


--2026-03-28 01:35:36--  ftp://pubftp-mte.itespp.org/Operational_Data/MTRF/2026/03/16
           => ‘/var/www/python/Qingcheng/.listing’
Resolving pubftp-mte.itespp.org (pubftp-mte.itespp.org)... 192.200.63.48
Connecting to pubftp-mte.itespp.org (pubftp-mte.itespp.org)|192.200.63.48|:21... connected.
Logging in as anonymous ... Logged in!
==> SYST ... done.    ==> PWD ... done.
==> TYPE I ... done.  ==> CWD (1) /Operational_Data/MTRF/2026/03 ... done.
==> PASV ... done.    ==> LIST ... done.

     0K .                                                       143M=0s

2026-03-28 01:35:36 (143 MB/s) - ‘/var/www/python/Qingcheng/.listing’ saved [1593]

Removed ‘/var/www/python/Qingcheng/.listing’.
--2026-03-28 01:35:36--  ftp://pubftp-mte.itespp.org/Operational_Data/MTRF/2026/03/16/16
           => ‘/var/www/python/Qingcheng/.listing’
==> CWD (1) /Operational_Data/MTRF/2026/03/16 ... done.
==> PASV ... done.    ==> LIST ... done.

     0K .                                                    


Latest file identified: OP-MTRF-202603160700.csv

--- Processing File: OP-MTRF-202603160700.csv ---
time of file is 20260316070000
                Interval      GMTIntervalEnd  Wind Forecast MW  \
259  03/17/2026 23:00:00 2026-03-18 04:00:00          23726.64   
260  03/17/2026 23:00:00 2026-03-18 04:00:00            560.14   
261  03/17/2026 22:00:00 2026-03-18 03:00:00          23041.40   
262  03/17/2026 22:00:00 2026-03-18 03:00:00            593.61   
263  03/17/2026 21:00:00 2026-03-18 02:00:00          21754.14   

     Solar Forecast MW   BAA         IntervalEnd  
259               0.00   SPP 2026-03-17 23:00:00  
260               0.00  SWPW 2026-03-17 23:00:00  
261               0.00   SPP 2026-03-17 22:00:00  
262               0.00  SWPW 2026-03-17 22:00:00  
263               1.02   SPP 2026-03-17 21:00:00  


PermissionError: [Errno 13] Permission denied: '/var/www/python/Qingcheng/'